# TMA multiplexed IF analysis — S100P and MHC II in LUAD

**Goal:** Quantify S100P and MHC II protein expression on PanCK+ tumor cells 
from a TMA and compare expression between IHC-classified MHC II+ and MHC II- patients.

**Input:** Cell-level object data exported from HALO after scanning a 4-plex 
fluorescence TMA (DAPI / PanCK-FITC / MHC II-TRITC / S100P-Cy5) on a Hamamatsu 
NanoZoomer scanner.

**Output:** Publication figures (main + supplemental) and patient-level summary tables.

## Setup

In [ ]:
# standard libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ceiba modules
from ceiba.tma_grid import (
    assign_grid_ids_unified,
    draw_safe_horizontal_lines,
    draw_safe_vertical_lines,
)
from ceiba.halo_utils import compute_cell_centers, rename_channels, aggregate_to_patient
from ceiba.plot_utils import make_comparison_figure, draw_ridgeline, sig_label, get_groups
# figure settings
sns.set(font_scale=1.6)
sns.set_style('ticks')
plt.rcParams['pdf.fonttype'] = 42  # embed fonts in PDF output — required by most journals
plt.rcParams['ps.fonttype'] = 42

In [ ]:
# paper-wide color palettes — consistent across all analyses
# MHC II pos = orange (#FF8811), MHC II neg = purple (#462255)
cmap_low_high = ['#462255FF', '#FF8811FF', '#9DD9D2FF', '#046E8FFF', '#D44D5CFF']  # neg first
cmap_high_low = ['#FF8811FF', '#462255FF', '#9DD9D2FF', '#046E8FFF', '#D44D5CFF']  # pos first
cmap_umap     = ['#9DD9D2FF', '#462255FF', '#FF8811FF', '#046E8FFF', '#D44D5CFF']  # used in scRNA/CosMx notebooks

## Data loading

In [ ]:
import yaml
from pathlib import Path

with open('/home/gh8sj/projects/mhc2-luad-paper/data/paths_config.yml') as f:
    cfg = yaml.safe_load(f)

repo_root = Path(cfg['repo_root'])

# data paths
tma_data_path = cfg['datasets']['tma_s100p']['raw']
metadata_path = cfg['datasets']['cosmx']['metadata']
classifications_path = cfg['datasets']['patient_classifications']['raw']

# output paths — resolved against repo root
fig_dir = repo_root / cfg['outputs']['figures']
table_dir = repo_root / cfg['outputs']['tables']

## Coordinate computation and channel renaming

In [ ]:
# load raw HALO cell-level export
full_df = pd.read_csv(tma_data_path)
print(f"Loaded {len(full_df):,} cells from {tma_data_path}")

# each unique image location corresponds to one physical TMA slide (4 total)
tmas = list(full_df['Image Location'].unique())

In [ ]:
# compute the midpoint from x and y min/max
full_df = compute_cell_centers(full_df)
y_col, x_col = 'YCenter', 'XCenter'


## TMA grid assignment

In [ ]:
# QC — visualize cell positions per TMA slide colored by HALO classifier label
# used to confirm orientation before grid assignment
for image_id in full_df['Image Location'].unique():
    figsize = (12, 4) if '15' in str(image_id) else (12, 8)
    plt.figure(figsize=figsize)
    sns.scatterplot(
        data=full_df[full_df['Image Location'] == image_id],
        x='XCenter', y='YCenter',
        hue='Classifier Label',
        s=10, alpha=0.6
    )
    plt.title(f'Cell positions — {image_id}')
    plt.gca().invert_yaxis()
    plt.legend(markerscale=5, loc='center left', bbox_to_anchor=(1, 0.5))
    plt.tight_layout()
    plt.show()

## TMA grid assignment and core identification

Each TMA image is processed independently to assign row/col grid positions to each cell.
Grid boundaries are detected automatically using `assign_grid_ids_unified`. Results are
verified visually by comparing core layout to H&E images of the TMA map.
Each TMA may require different rotation/flip parameters depending on scan orientation —
these were determined manually for each slide.

**Scan order to physical TMA mapping** (confirmed by comparing filenames to H&E TMA maps):

| Index | Physical TMA | Scan time |
|-------|-------------|-----------|
| tmas[0] | TMA 4 | 2025-06-26 15:31 |
| tmas[1] | TMA 2 | 2025-06-26 16:03 |
| tmas[2] | TMA 1 | 2025-06-26 14:50 |
| tmas[3] | TMA 3 | 2025-06-26 13:51 |

In [ ]:
# TMA 1 — rotated scan, PCA reorientation applied
# tmas[2] corresponds to physical TMA 1 (scan order != TMA number)
tma1 = full_df.loc[full_df['Image Location'] == tmas[2]].copy()

tma1, rows, cols = assign_grid_ids_unified(
    tma1,
    x_col='XCenter',
    y_col='YCenter',
    clearance=100,
    min_spacing=3000,
    horizontal_func=draw_safe_horizontal_lines,
    vertical_func=draw_safe_vertical_lines,
    rotate_horizontal=True,
    rotate_vertical=True,
    plot=True
)

# remove cores with fewer than 50 cells
core_counts = tma1['Core_ID'].value_counts()
valid_cores = core_counts[core_counts >= 50].index
tma1 = tma1[tma1['Core_ID'].isin(valid_cores)]

# re-index rows and cols to be contiguous after filtering
for dim in ['row', 'col']:
    sorted_vals = sorted(tma1[dim].dropna().unique())
    remap = {v: i for i, v in enumerate(sorted_vals)}
    tma1[dim] = tma1[dim].map(remap)
tma1['Core_ID'] = 'Row_' + tma1['row'].astype(str) + '_Col_' + tma1['col'].astype(str)

# verify orientation
fig, axes = plt.subplots(1, 2, figsize=(12, 7))
for ax, hue, title in zip(axes, ['row', 'col'], ['rows', 'cols']):
    sns.scatterplot(data=tma1, x=y_col, y=x_col, s=5, hue=hue, palette='tab10', ax=ax)
    ax.set_title(title)
    ax.invert_yaxis()
    ax.legend(loc='center left', bbox_to_anchor=(1, 0.5), title=hue.capitalize())
plt.tight_layout()
plt.show()

In [ ]:
# TMA 2 — slightly rotated scan, requires PCA reorientation and reduced min_spacing
# initial axis-aligned attempt failed; density method + rotation used instead
tma2 = full_df.loc[full_df['Image Location'] == tmas[1]].copy()

tma2, rows, cols = assign_grid_ids_unified(
    tma2,
    x_col='XCenter',
    y_col='YCenter',
    clearance=100,
    min_spacing=2500,
    horizontal_func=draw_safe_horizontal_lines,
    vertical_func=lambda d, **kwargs: draw_safe_vertical_lines(
        d, method='density', plot=True, **kwargs
    ),
    rotate_horizontal=True,
    rotate_vertical=True,
    plot=True
)

# remove cores with fewer than 50 cells — excludes control row and empty cores
# threshold of 50 was determined by visual inspection of core cellularity
core_counts = tma2['Core_ID'].value_counts()
valid_cores = core_counts[core_counts >= 50].index
tma2 = tma2[tma2['Core_ID'].isin(valid_cores)]

# re-index rows and cols to be contiguous after filtering
# re-index rows and cols to be contiguous after filtering
for dim in ['row', 'col']:
    sorted_vals = sorted(tma2[dim].dropna().unique())
    remap = {v: i for i, v in enumerate(sorted_vals)}
    tma2[dim] = tma2[dim].map(remap)

# flip column order to match H&E orientation (scan is mirrored left-right)
n_cols = tma2['col'].max()
tma2['col'] = n_cols - tma2['col']

tma2['Core_ID'] = 'Row_' + tma2['row'].astype(str) + '_Col_' + tma2['col'].astype(str)

# verify orientation against H&E ground truth — x-axis is flipped relative to scan
# col_flipped will be assigned during metadata merge to correct for this
fig, axes = plt.subplots(1, 2, figsize=(12, 7))
for ax, hue, title in zip(axes, ['row', 'col'], ['rows', 'cols']):
    sns.scatterplot(data=tma2, x=y_col, y=x_col, s=5, hue=hue, palette='tab10', ax=ax)
    ax.set_title(title)
    ax.invert_yaxis()
    ax.invert_xaxis()
    ax.legend(loc='center left', bbox_to_anchor=(1, 0.5), title=hue.capitalize())
plt.tight_layout()
plt.show()

In [ ]:
# TMA 3 — axis-aligned, no rotation needed
tma3 = full_df.loc[full_df['Image Location'] == tmas[3]].copy()

tma3, rows, cols = assign_grid_ids_unified(
    tma3,
    x_col='XCenter',
    y_col='YCenter',
    clearance=100,
    min_spacing=3000,
    horizontal_func=draw_safe_horizontal_lines,
    vertical_func=draw_safe_vertical_lines,
    plot=True
)

# verify row assignment visually
plt.figure(figsize=(5, 7))
sns.scatterplot(data=tma3, x=y_col, y=x_col, s=5, hue='row', palette='tab10')
plt.legend(loc='center left', bbox_to_anchor=(1, 0.5), title='Row')
plt.gca().invert_yaxis()
plt.gca().invert_xaxis()
plt.tight_layout()
plt.show()

In [ ]:
# TMA 4 — axis-aligned, no rotation needed
tma4 = full_df.loc[full_df['Image Location'] == tmas[0]].copy()

tma4, rows, cols = assign_grid_ids_unified(
    tma4,
    x_col='XCenter',
    y_col='YCenter',
    clearance=100,
    min_spacing=3000,
    horizontal_func=draw_safe_horizontal_lines,
    vertical_func=draw_safe_vertical_lines,
    plot=True
)

# remove cores with fewer than 50 cells
core_counts = tma4['Core_ID'].value_counts()
valid_cores = core_counts[core_counts >= 50].index
tma4 = tma4[tma4['Core_ID'].isin(valid_cores)]

# re-index rows and cols to be contiguous after filtering
for dim in ['row', 'col']:
    sorted_vals = sorted(tma4[dim].dropna().unique())
    remap = {v: i for i, v in enumerate(sorted_vals)}
    tma4[dim] = tma4[dim].map(remap)

# flip column order to match H&E orientation
n_cols = tma4['col'].max()
tma4['col'] = n_cols - tma4['col']

tma4['Core_ID'] = 'Row_' + tma4['row'].astype(str) + '_Col_' + tma4['col'].astype(str)

# verify orientation
fig, axes = plt.subplots(1, 2, figsize=(12, 7))
for ax, hue, title in zip(axes, ['row', 'col'], ['rows', 'cols']):
    sns.scatterplot(data=tma4, x=y_col, y=x_col, s=5, hue=hue, palette='tab10', ax=ax)
    ax.set_title(title)
    ax.invert_yaxis()
    ax.invert_xaxis()
    ax.legend(loc='center left', bbox_to_anchor=(1, 0.5), title=hue.capitalize())
plt.tight_layout()
plt.show()

## Metadata merge

## Combining TMAs and linking to patient metadata

All four TMA DataFrames are combined into a single object, then merged with
patient-level metadata to assign MHC II classification, region, and PatientID
to each cell.

| TMA | Patient IDs | Notes |
|-----|-------------|-------|
| TMA 1 | 1–26 | 26 patients, 2 × 13 rows |
| TMA 2 | 27–52 | 26 patients, 2 × 13 rows |
| TMA 3 | 53–78 | 26 patients, 2 × 13 rows |
| TMA 4 | 79–91 | 13 patients, partial TMA (1 × 13 rows) |

In [ ]:
# combine all four TMAs into a single DataFrame
tma_list = [tma1, tma2, tma3, tma4]

# final orientation check across all TMAs before merging
for tma in tma_list:
    fig, ax = plt.subplots(figsize=(5, 7))
    sns.scatterplot(data=tma, x=y_col, y=x_col, s=5, hue='row', palette='tab10', ax=ax)
    ax.invert_yaxis()
    ax.invert_xaxis()
    ax.legend(loc='center left', bbox_to_anchor=(1, 0.5), title='Row')
    plt.tight_layout()
    plt.show()

In [ ]:
# load patient classification metadata
patient_classifications = pd.read_csv(
    cfg['datasets']['patient_classifications']['raw'],
    index_col=0
)
patient_classifications.loc['tonsil'] = ['Tonsil', 'Tonsil', 'Tonsil']
patient_classifications['patient'] = patient_classifications.index.astype(str)

# load CosMx metadata — TMA position (slide/fov/patient/region) per core
metadata = pd.read_csv(
    cfg['datasets']['cosmx']['metadata'],
    index_col=0
)

In [ ]:
# --- Patient ID and region assignment ---
# Each TMA has a different layout. Patient IDs are assigned based on row/col position.
# Columns are grouped in triplets (CT, PT, N) per patient.
# Start IDs: TMA1=1, TMA2=27, TMA3=53, TMA4=78

def assign_patient_ids_tma1_or_2(df, start_patient_id=1, exclude_control_row=None, reverse_cols=False):
    """
    Assign PatientID and region for TMA 1 or TMA 2.
    Patients increase top to bottom, cores grouped in column triplets (CT/PT/N).
    """
    df = df.copy()
    unique_rows = sorted(df['row'].unique())
    unique_cols = sorted(df['col'].unique(), reverse=reverse_cols)
    if exclude_control_row is not None:
        df = df[df['row'] != exclude_control_row].copy()
        unique_rows = [r for r in unique_rows if r != exclude_control_row]
    row_map = {r: i for i, r in enumerate(unique_rows)}
    col_map = {c: i for i, c in enumerate(unique_cols)}
    df['row_idx'] = df['row'].map(row_map)
    df['col_idx'] = df['col'].map(col_map)
    n_rows = len(unique_rows)
    df['PatientID'] = df.apply(
        lambda row: start_patient_id + row['row_idx'] + (row['col_idx'] // 3) * n_rows,
        axis=1
    )
    return df


def assign_patient_ids_tma3(df, start_patient_id=53, exclude_top_row=True):
    df = df.copy()
    
    # exclude gap columns
    valid_cols = [0, 1, 2, 4, 5, 6]
    df = df[df['col'].isin(valid_cols)].copy()
    
    unique_rows = sorted(df['row'].unique())
    if exclude_top_row:
        control_row = unique_rows[0]
        df = df[df['row'] != control_row].copy()
        unique_rows = [r for r in unique_rows if r != control_row]

    # reverse col order so patient numbering starts top-left
    used_cols = sorted(df['col'].unique(), reverse=True)
    row_map = {val: i for i, val in enumerate(unique_rows)}
    col_map = {val: i for i, val in enumerate(used_cols)}

    df['row_idx'] = df['row'].map(row_map)
    df['col_idx'] = df['col'].map(col_map)

    n_rows = len(unique_rows)
    df['PatientID'] = df.apply(
        lambda row: start_patient_id + row['row_idx'] + (row['col_idx'] // 3) * n_rows,
        axis=1
    )
    return df


def assign_patient_ids_tma4(df, start_patient_id=79, exclude_top=True):
    """
    Assign PatientID for TMA 4.
    Only 3 columns (CT/PT/N), patients increase top to bottom.
    Top row is a control and is excluded.
    """
    df = df.copy()
    unique_rows = sorted(df['row'].unique())
    if exclude_top:
        unique_rows = unique_rows[1:]
        df = df[df['row'].isin(unique_rows)].copy()
    row_map = {v: i for i, v in enumerate(unique_rows)}
    df['row_idx'] = df['row'].map(row_map)
    df['PatientID'] = df['row_idx'].apply(lambda r: start_patient_id + r)
    return df


# apply assignment functions
tma_list[0] = assign_patient_ids_tma1_or_2(tma1, start_patient_id=1,  reverse_cols=True)
tma_list[1] = assign_patient_ids_tma1_or_2(tma2, start_patient_id=27, reverse_cols=False)
tma_list[2] = assign_patient_ids_tma3(tma3, start_patient_id=53)
tma_list[3] = assign_patient_ids_tma4(tma4, start_patient_id=79)

# assign region from column position — defined per TMA based on H&E map
# each patient has 3 cores: CT (central tumor), PT (peripheral tumor), N (normal adjacent)
region_maps = {
    0: {0: 'N', 1: 'PT', 2: 'CT', 3: 'N', 4: 'PT', 5: 'CT'},  # TMA 1
    1: {0: 'CT', 1: 'PT', 2: 'N', 3: 'CT', 4: 'PT', 5: 'N'},  # TMA 2
    2: {6: 'CT', 5: 'PT', 4: 'N', 2: 'CT', 1: 'PT', 0: 'N'},  # TMA 3
    3: {0: 'CT', 1: 'PT', 2: 'N'},                              # TMA 4
}
for i, tma in enumerate(tma_list):
    tma['Region'] = tma['col'].map(region_maps[i])
    tma['tma'] = i + 1

# verify PatientID assignment
for tma in tma_list:
    fig, axes = plt.subplots(1, 2, figsize=(14, 12))
    
    # patient ID
    sns.scatterplot(data=tma, x=y_col, y=x_col, s=3, hue='PatientID',
                    palette='tab10', ax=axes[0], legend=True)
    axes[0].set_title('patient ID', fontsize=18)
    axes[0].invert_yaxis()
    axes[0].invert_xaxis()
    legend = axes[0].get_legend()
    handles, labels = legend.legend_handles, [t.get_text() for t in legend.get_texts()]
    legend.remove()
    axes[0].legend(handles, labels,
                   loc='upper center', bbox_to_anchor=(0.5, -0.1),
                   title='PatientID', markerscale=5, fontsize=12,
                   ncol=6, frameon=False)

    # region
    sns.scatterplot(data=tma, x=y_col, y=x_col, s=3, hue='Region',
                    palette='tab10', ax=axes[1])
    axes[1].set_title('region')
    axes[1].invert_yaxis()
    axes[1].invert_xaxis()
    axes[1].legend(loc='upper center', bbox_to_anchor=(0.5, -0.1),
                   ncol=3, markerscale=5, title='Region', frameon=False)

    plt.suptitle(tma.iloc[0, 0], fontsize=18)
    plt.tight_layout()
    plt.show()

In [ ]:
# combine all TMAs and merge with patient classifications
tma_df = pd.concat(tma_list, axis=0, ignore_index=True)

# drop unassigned cells (outside grid or control rows)
tma_df = tma_df[tma_df['PatientID'].notna()].copy()
tma_df['PatientID'] = tma_df['PatientID'].astype(int)

# merge with IHC classifications — excludes tonsil control
patient_classifications = patient_classifications[
    patient_classifications['patient'] != 'tonsil'
].copy()
patient_classifications['patient'] = patient_classifications['patient'].astype(int)

merged_df = pd.merge(
    tma_df, patient_classifications,
    left_on='PatientID', right_on='patient',
    how='left'
)

print(f"Total cells:     {len(merged_df):,}")
print(f"Patients:        {merged_df['PatientID'].nunique()}")
print(f"Classifications: {merged_df['patient classification'].value_counts().to_dict()}")

unmatched = merged_df['patient classification'].isna().sum()
if unmatched > 0:
    print(f"\nWARNING: {unmatched:,} cells have no classification match — check PatientID mapping")

In [ ]:
# verify patient classification assignment — one plot per TMA
classification_palette = {
    'class II positive':  cmap_high_low[0],
    'class II negative':  cmap_high_low[1],
    'class II clonal':    cmap_high_low[2],
    'no malignant cells': cmap_high_low[3]
}

for tma_id in sorted(merged_df['tma'].unique()):
    fig, ax = plt.subplots(figsize=(7, 7))
    sns.scatterplot(
        data=merged_df.loc[merged_df['tma'] == tma_id],
        x=y_col, y=x_col, s=5,
        hue='patient classification',
        palette=classification_palette,
        ax=ax
    )
    ax.invert_yaxis()
    ax.invert_xaxis()
    ax.legend(loc='center left', bbox_to_anchor=(1, 0.5), markerscale = 5,title='patient classification')
    ax.set_title(f'TMA {tma_id}')
    plt.tight_layout()
    plt.show()

In [ ]:
# --- patient ID and merge diagnostics ---
print(f"Total patients: {merged_df['PatientID'].nunique()}")
print(f"Patient ID range: {merged_df['PatientID'].min()} to {merged_df['PatientID'].max()}")
print(f"\nCells with NaN region: {merged_df['Region'].isna().sum()}")
print(f"Classifications:\n{merged_df['patient classification'].value_counts(dropna=False)}")

core_counts = merged_df.groupby('PatientID')['Core_ID'].nunique()
print(f"\nPatients with fewer than 3 cores:")
print(core_counts[core_counts < 3])

unmatched = merged_df['patient classification'].isna().sum()
if unmatched > 0:
    print(f"\nWARNING: {unmatched:,} cells have no classification match")

## QC — channel intensity distributions

Overlapping histograms of positive vs negative cell intensity for each marker,
used to assess HALO positivity thresholds. Log-scale plots reveal distributional
structure not visible on linear scale.

**Key findings:**
- MHC II: well separated, HALO thresholds appear reasonable
- S100P and PanCK: bimodal distributions with overlap at threshold — investigated further below
- PanCK distributions are similar across CT, PT, and NAT regions, consistent with
  normal epithelial cells present in all three core types
- No evidence of channel spillover in PanCK+ tumor cells (r=-0.11 vs r=0.59 in PanCK-)

**TMA-level notes:**
- TMA 4: ~26% of cells have MHC II intensity = 0. May reflect lower staining efficiency;
  included in linear plots, excluded from log-scale plots
- TMA 2: rightward shift in S100P intensity relative to other TMAs. Binary positivity
  calls partially mitigate this; sensitivity analysis excluding TMA 2 showed stronger
  results (see statistical comparisons section)

In [ ]:
from scipy import stats

# rename channels for readability
df = rename_channels(merged_df, {
    'Cy5 635':  'S100P',
    'Fitc 474': 'PanCK',
    'Trtc 554': 'MHC II',
})
stains = ['MHC II', 'S100P', 'PanCK']
log_floor = 0.01
tma_colors = {1: 'steelblue', 2: 'salmon', 3: 'mediumseagreen', 4: 'mediumpurple'}
region_colors = {'N': 'gray', 'CT': 'steelblue', 'PT': 'salmon'}

In [ ]:
# 1. intensity distributions by TMA
sns.set(font_scale=1.2)
sns.set_style('ticks')
fig, axes = plt.subplots(3, 2, figsize=(16, 12))
plt.subplots_adjust(left=0.05, right=0.98, top=0.95, bottom=0.07, hspace=0.45, wspace=0.3)
for row, stain in enumerate(stains):
    intensity_col = f'{stain} Avg Cell Intensity'
    if intensity_col not in df.columns:
        continue
    lin_data = df[['tma', intensity_col]].copy()
    lin_data['intensity'] = lin_data[intensity_col]
    ax_lin = axes[row, 0]
    ax_lin.set_visible(False)
    draw_ridgeline(fig, ax_lin.get_position(), lin_data, 'intensity',
                   group_col='tma', color_map=tma_colors,
                   xlabel='Average cell intensity',
                   title=f'{stain} — linear', log=False)
    log_data = df.loc[df[intensity_col] > log_floor, ['tma', intensity_col]].copy()
    log_data['log_intensity'] = np.log10(log_data[intensity_col])
    ax_log = axes[row, 1]
    ax_log.set_visible(False)
    draw_ridgeline(fig, ax_log.get_position(), log_data, 'log_intensity',
                   group_col='tma', color_map=tma_colors,
                   xlabel='Average cell intensity (log scale)',
                   title=f'{stain} — log scale', log=True)
plt.show()
sns.set(font_scale=1.6)
sns.set_style('ticks')

In [ ]:
# 2. PanCK intensity by region
sns.set(font_scale=1.2)
sns.set_style('ticks')
region_colors_ordered = [('N', 'gray'), ('CT', 'steelblue'), ('PT', 'salmon')]
region_color_map = {'N': 'gray', 'CT': 'steelblue', 'PT': 'salmon'}
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plt.subplots_adjust(left=0.05, right=0.98, top=0.88, bottom=0.12, wspace=0.3)
ax_lin = axes[0]
all_vals = df['PanCK Avg Cell Intensity']
bins = np.linspace(0, all_vals.max(), 50)
for region, color in region_colors_ordered:
    vals = df.loc[df['Region'] == region, 'PanCK Avg Cell Intensity']
    ax_lin.hist(vals, bins=bins, alpha=0.5, density=True, label=region, color=color)
ax_lin.set_xlabel('Average cell intensity')
ax_lin.set_ylabel('Density')
ax_lin.set_title('PanCK — linear')
ax_lin.legend()
ax_lin.spines[['top', 'right']].set_visible(False)
plot_data = df.loc[df['PanCK Avg Cell Intensity'] > log_floor,
                   ['Region', 'PanCK Avg Cell Intensity']].copy()
plot_data['log_intensity'] = np.log10(plot_data['PanCK Avg Cell Intensity'])
ax_log = axes[1]
ax_log.set_visible(False)
draw_ridgeline(fig, ax_log.get_position(), plot_data, 'log_intensity',
               group_col='Region', color_map=region_color_map,
               xlabel='Average cell intensity (log scale)',
               title='PanCK — log scale', log=True)
fig.suptitle('PanCK intensity by region', fontsize=11, y=1.0)
plt.show()
sns.set(font_scale=1.6)
sns.set_style('ticks')

In [ ]:
# 3. NAT quantiles
nat_panck = df.loc[df['Region'] == 'N', 'PanCK Avg Cell Intensity']
print(nat_panck.describe())
print("95th percentile:", nat_panck.quantile(0.95))
print("99th percentile:", nat_panck.quantile(0.99))

In [ ]:
# 4. spillover check — PanCK vs MHC II correlation
# no spillover in PanCK+ cells (r=-0.11) vs PanCK- cells (r=0.59)
panck_pos = df[df['PanCK Positive Cells'] == 1].copy()
panck_neg = df[df['PanCK Positive Cells'] == 0].copy()
r, p = stats.pearsonr(panck_pos['PanCK Avg Cell Intensity'],
                       panck_pos['MHC II Avg Cell Intensity'])
print(f"PanCK+ cells — PanCK vs MHC II: r={r:.3f}, p={p:.2e}")
r2, p2 = stats.pearsonr(panck_neg['PanCK Avg Cell Intensity'],
                         panck_neg['MHC II Avg Cell Intensity'])
print(f"PanCK- cells — PanCK vs MHC II: r={r2:.3f}, p={p2:.2e}")

## QC — cell phenotype composition

Stacked bar plots showing cell phenotype composition per patient, built from
per-cell binary positivity flags (more reliable than HALO's multiplex phenotype
counts which showed sparse population in many cores).

Two views:
1. All cells — shows the full TME composition including PanCK- populations
2. PanCK+ cells only — isolates tumor cell phenotypes for the main analysis

These are not publication figures. Use to identify outlier patients or
TMA-level batch effects before analysis.

In [ ]:
# all cells — full TME composition including PanCK- populations
# PanCK+S100P-MHCII- (light green) dominates most patients — tumor cells lacking both markers
# PanCK-S100P-MHCII+ (red) = immune/stromal MHC II+ cells (likely APCs, macrophages)
# PanCK-S100P-MHCII- (gray) = triple negative stromal/immune cells

df_all = merged_df[merged_df['Region'] != 'N'].copy()

conditions = {
    'PanCK+S100P+MHCII+': (df_all['Fitc 474 Positive Cells'] == 1) & (df_all['Cy5 635 Positive Cells'] == 1) & (df_all['Trtc 554 Positive Cells'] == 1),
    'PanCK+S100P+MHCII-': (df_all['Fitc 474 Positive Cells'] == 1) & (df_all['Cy5 635 Positive Cells'] == 1) & (df_all['Trtc 554 Positive Cells'] == 0),
    'PanCK+S100P-MHCII+': (df_all['Fitc 474 Positive Cells'] == 1) & (df_all['Cy5 635 Positive Cells'] == 0) & (df_all['Trtc 554 Positive Cells'] == 1),
    'PanCK+S100P-MHCII-': (df_all['Fitc 474 Positive Cells'] == 1) & (df_all['Cy5 635 Positive Cells'] == 0) & (df_all['Trtc 554 Positive Cells'] == 0),
    'PanCK-S100P+MHCII+': (df_all['Fitc 474 Positive Cells'] == 0) & (df_all['Cy5 635 Positive Cells'] == 1) & (df_all['Trtc 554 Positive Cells'] == 1),
    'PanCK-S100P+MHCII-': (df_all['Fitc 474 Positive Cells'] == 0) & (df_all['Cy5 635 Positive Cells'] == 1) & (df_all['Trtc 554 Positive Cells'] == 0),
    'PanCK-S100P-MHCII+': (df_all['Fitc 474 Positive Cells'] == 0) & (df_all['Cy5 635 Positive Cells'] == 0) & (df_all['Trtc 554 Positive Cells'] == 1),
    'PanCK-S100P-MHCII-': (df_all['Fitc 474 Positive Cells'] == 0) & (df_all['Cy5 635 Positive Cells'] == 0) & (df_all['Trtc 554 Positive Cells'] == 0),

}

for label, mask in conditions.items():
    df_all[label] = mask.astype(int)

phenotype_cols = list(conditions.keys())

proportions = (
    df_all.groupby('PatientID')[phenotype_cols]
    .sum()
    .pipe(lambda d: d.div(d.sum(axis=1), axis=0))
    .sort_index()
)

fig, ax = plt.subplots(figsize=(18, 6))
proportions.plot(kind='bar', stacked=True, width=0.8, ax=ax,
                 color=['#2166ac', '#92c5de', '#4dac26', '#b8e186',
                        '#7b3294', '#c2a5cf', '#d7191c', '#b0b0b0'
])
ax.set_xticks(range(len(proportions)))
ax.set_xticklabels(proportions.index, rotation=90, fontsize=7)
ax.set_ylabel('Fraction of all cells')
ax.set_xlabel('Patient ID')
ax.set_title('Cell phenotype composition per patient (all cells)')
ax.legend(title='Cell type', bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# PanCK+ tumor cells only — phenotype composition for main analysis population
# most patients dominated by PanCK+S100P-MHCII- (light green) — double negative tumor cells
# variation in S100P+ and MHCII+ fractions visible across patients
df_tumor = merged_df[
    (merged_df['Fitc 474 Positive Cells'] == 1) &
    (merged_df['Region'] != 'N')
].copy()

conditions = {
    'PanCK+S100P+MHCII+': (df_tumor['Cy5 635 Positive Cells'] == 1) & (df_tumor['Trtc 554 Positive Cells'] == 1),
    'PanCK+S100P+MHCII-': (df_tumor['Cy5 635 Positive Cells'] == 1) & (df_tumor['Trtc 554 Positive Cells'] == 0),
    'PanCK+S100P-MHCII+': (df_tumor['Cy5 635 Positive Cells'] == 0) & (df_tumor['Trtc 554 Positive Cells'] == 1),
    'PanCK+S100P-MHCII-': (df_tumor['Cy5 635 Positive Cells'] == 0) & (df_tumor['Trtc 554 Positive Cells'] == 0),
}

for label, mask in conditions.items():
    df_tumor[label] = mask.astype(int)

phenotype_cols = list(conditions.keys())

proportions = (
    df_tumor.groupby('PatientID')[phenotype_cols]
    .sum()
    .pipe(lambda d: d.div(d.sum(axis=1), axis=0))
    .sort_index()
)

fig, ax = plt.subplots(figsize=(18, 6))
proportions.plot(kind='bar', stacked=True, width=0.8, ax=ax,
                 color=['#2166ac', '#92c5de', '#4dac26', '#b8e186'])
ax.set_xticks(range(len(proportions)))
ax.set_xticklabels(proportions.index, rotation=90, fontsize=7)
ax.set_ylabel('Fraction of PanCK+ cells')
ax.set_xlabel('Patient ID')
ax.set_title('PanCK+ cell phenotype composition per patient')
ax.legend(title='Cell type', bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# --- patient-level aggregation ---
cell_types = [
    'PanCK+S100P+MHCII+ Cells',
    'PanCK+S100P+MHCII- Cells',
    'PanCK+S100P-MHCII+ Cells',
    'PanCK+S100P-MHCII- Cells',
    'PanCK-S100P+MHCII+ Cells',
    'PanCK-S100P+MHCII- Cells',
    'PanCK-S100P-MHCII+ Cells',
]

patient_agg = aggregate_to_patient(
    merged_df,
    cell_type_cols=cell_types,
    exclude_regions=['N']
)

plot_df = patient_agg[
    patient_agg['patient classification'].isin(['class II positive', 'class II negative'])
].copy()

print(f"Patients in analysis: {len(plot_df)}")
print(plot_df['patient classification'].value_counts())

# Figures and tables

In [ ]:
fig_out = fig_dir / 'figure4'
table_out = table_dir / 'figure4'
fig_out.mkdir(parents=True, exist_ok=True)
table_out.mkdir(parents=True, exist_ok=True)

## Statistical comparisons — figure setup

In [ ]:
# create output directories if they don't exist
from pathlib import Path
Path('outputs/figures/figure4').mkdir(parents=True, exist_ok=True)
Path('outputs/tables/figure4').mkdir(parents=True, exist_ok=True)

In [ ]:
from scipy.stats import mannwhitneyu

sns.set(font_scale=1.2)
sns.set_style('ticks')
NEG_COLOR = '#462255'
POS_COLOR = '#FF8811'

sig_types = [
    'PanCK+S100P+MHCII- Cells',
    'PanCK-S100P+MHCII- Cells',
]
supp_types = [
    'PanCK+S100P+MHCII+ Cells',
    'PanCK+S100P-MHCII+ Cells',
    'PanCK+S100P-MHCII- Cells',
    'PanCK-S100P+MHCII+ Cells',
    'PanCK-S100P-MHCII+ Cells',
]

def draw_panel(ax, ct, neg, pos, pval):
    bp = ax.boxplot(
        [neg, pos], positions=[0, 1], widths=0.35, patch_artist=True,
        medianprops=dict(linewidth=2), whiskerprops=dict(linewidth=1),
        capprops=dict(linewidth=1), showfliers=False
    )
    for patch, color in zip(bp['boxes'], [NEG_COLOR, POS_COLOR]):
        patch.set_facecolor('none'); patch.set_edgecolor(color); patch.set_linewidth(1.5)
    for median, color in zip(bp['medians'], [NEG_COLOR, POS_COLOR]):
        median.set_color(color); median.set_linewidth(2)
    for whisker, color in zip(bp['whiskers'], [NEG_COLOR, NEG_COLOR, POS_COLOR, POS_COLOR]):
        whisker.set_color(color)
    for cap, color in zip(bp['caps'], [NEG_COLOR, NEG_COLOR, POS_COLOR, POS_COLOR]):
        cap.set_color(color)
    for data, x, color in zip([neg, pos], [0, 1], [NEG_COLOR, POS_COLOR]):
        jitter = np.random.uniform(-0.12, 0.12, size=len(data))
        ax.scatter(x + jitter, data, color=color, s=18, alpha=0.8, zorder=3, linewidths=0)
    ax.set_yscale('log')
    ax.set_xticks([0, 1])
    ax.set_xticklabels(['MHC II\nneg', 'MHC II\npos'])
    ax.set_title(ct.replace(' Cells', ''), pad=6)
    ax.text(0.5, 0.97, f'p = {pval:.2e} ({sig_label(pval)})',
            transform=ax.transAxes, ha='center', va='top')
    ax.spines[['top', 'right']].set_visible(False)

def plot_phenotype_comparison(types, layout, figsize, source_df=None, title=None, savepath=None):
    if source_df is None:
        source_df = plot_df
    nrow, ncol = layout
    fig, axes = plt.subplots(nrow, ncol, figsize=figsize)
    axes = np.array(axes).flatten()
    all_vals = np.concatenate([np.concatenate(get_groups(ct, source_df)) for ct in types])
    ymin = all_vals[all_vals > 0].min() * 0.5
    ymax = all_vals.max() * 4
    for ax, ct in zip(axes, types):
        neg, pos = get_groups(ct, source_df)
        _, pval = mannwhitneyu(neg, pos, alternative='two-sided')
        draw_panel(ax, ct, neg, pos, pval)
        ax.set_ylim(ymin, ymax)
    for i, ax in enumerate(axes[:len(types)]):
        ax.set_ylabel('Fraction of total cells' if i % ncol == 0 else '')
    for ax in axes[len(types):]:
        ax.set_visible(False)
    if title:
        fig.suptitle(title, fontsize=10, y=1.02)
    plt.tight_layout()
    if savepath:
        fig.savefig(savepath, bbox_inches='tight', dpi=300)
    plt.show()

## Statistical comparisons — main figure

In [ ]:
#make_figure(sig_types, layout=(1, 2), figsize=(7, 5))
plot_phenotype_comparison(sig_types, layout=(1, 2), figsize=(7, 5),
                          savepath=fig_out / 'figure4c.pdf')

## Statistical comparisons — supplemental figure

In [ ]:
#make_figure(supp_types, layout=(2, 3), figsize=(10, 7))
plot_phenotype_comparison(supp_types, layout=(2, 3), figsize=(10, 7),
                          savepath=fig_out / 'supplemental_figure.pdf')

## Sensitivity analysis — TMA 2 excluded

TMA 2 showed elevated S100P staining intensity relative to other slides.
Results strengthen when TMA 2 is excluded, confirming the finding is not a batch artifact.

| Comparison | Full cohort | TMA 2 excluded |
|---|---|---|
| PanCK+S100P+MHCII- | p=6.51e-04 (***) | p=2.00e-04 (***) |
| PanCK-S100P+MHCII- | p=5.95e-04 (***) | p=7.54e-05 (***) |

In [ ]:
plot_df_no_tma2 = plot_df[~plot_df['PatientID'].between(27, 52)].copy()
plot_phenotype_comparison(sig_types, layout=(1, 2), figsize=(7, 5),
            source_df=plot_df_no_tma2,
            title='Sensitivity check — TMA 2 excluded')

## Summary tables

In [ ]:
# patient-level phenotype fractions — main analysis groups only
summary = plot_df[
    ['PatientID', 'patient classification', 'Total_Cells'] +
    [f'{ct}_fraction' for ct in cell_types]
].copy()
summary.columns = summary.columns.str.replace('_fraction', '', regex=False)
summary.to_csv(table_out / 'patient_phenotype_fractions.csv', index=False)
print(f"Saved {len(summary)} patients → {table_out / 'patient_phenotype_fractions.csv'}")

# sensitivity check — TMA 2 excluded
summary_no_tma2 = plot_df_no_tma2[
    ['PatientID', 'patient classification', 'Total_Cells'] +
    [f'{ct}_fraction' for ct in cell_types]
].copy()
summary_no_tma2.columns = summary_no_tma2.columns.str.replace('_fraction', '', regex=False)
summary_no_tma2.to_csv(table_out / 'patient_phenotype_fractions_no_tma2.csv', index=False)
print(f"Saved {len(summary_no_tma2)} patients → {table_out / 'patient_phenotype_fractions_no_tma2.csv'}")